# CanvasXpress Notebook Renderer — R Demo

> ⚠️ **Select the R kernel before running.** Click the kernel picker at the top-right of this
> notebook and choose **R** (the `ir` kernelspec). If it runs on a Python kernel you'll get
> `SyntaxError` on the first R line (`spec <- ...`) — that means the wrong kernel is selected.
>
> No R kernel listed? Install it once, then reopen the picker:
> ```r
> install.packages(c("IRkernel", "jsonlite"))
> IRkernel::installspec()   # registers the "R" kernel with Jupyter/VS Code
> ```

The same demo as `canvasxpress-demo.ipynb`, but driven from an **R (IRkernel)** kernel.

The renderer only cares about the mime type `application/canvasxpress+json` — not which
language produced it. From R we serialize a `list(data=, config=)` spec to a JSON string
with `jsonlite::toJSON(..., auto_unbox = TRUE)` and publish it with
`IRdisplay::publish_mimebundle()`.

> This is **not** the same as the `canvasXpress()` function from the R package — that returns
> an htmlwidget (mime type `text/html`), which the VS Code renderer does not handle. Here we
> emit the raw `{data, config}` spec, which is the exact structure the R package feeds to the
> JavaScript engine anyway.

## A tiny helper

`show()` serializes a spec and publishes it under the renderer's mime type. `auto_unbox = TRUE`
keeps scalars from becoming 1-element arrays.

In [ ]:
library(jsonlite)

show <- function(data, config, ...) {
  spec <- c(list(data = data, config = config), list(...))  # optional: events, width, height
  json <- toJSON(spec, auto_unbox = TRUE, null = "null")
  IRdisplay::publish_mimebundle(list("application/canvasxpress+json" = json))
}

## 1. Bar chart

In [ ]:
show(
  data = list(y = list(
    vars = list("Revenue"),
    smps = list("Q1", "Q2", "Q3", "Q4"),
    data = list(c(10, 14, 9, 17))
  )),
  config = list(
    graphType = "Bar", title = "Quarterly Revenue",
    xAxis = list("Revenue"), smpTitle = "Quarter", showLegend = FALSE
  )
)

## 2. Scatter plot

For `Scatter2D`, `smps` are the axis dimensions and `vars` are the point ids (one data row per point).

In [ ]:
show(
  data = list(y = list(
    smps = list("Weight", "MPG"),
    vars = as.list(paste0("c", 1:8)),
    data = list(c(3.2,24), c(2.8,30), c(4.1,18), c(3.6,21),
                c(2.4,33), c(3.9,19), c(2.1,36), c(4.5,15))
  )),
  config = list(
    graphType = "Scatter2D", title = "Weight vs. MPG",
    xAxis = list("Weight"), yAxis = list("MPG")
  )
)

## 3. Heatmap

In [ ]:
show(
  data = list(y = list(
    vars = list("Gene A", "Gene B", "Gene C", "Gene D"),
    smps = list("S1", "S2", "S3", "S4", "S5", "S6"),
    data = list(
      c(1.2, -0.4, 0.8, 1.9, -1.1, 0.3),
      c(-0.9, 1.4, 0.2, -0.7, 1.8, -0.5),
      c(0.5, 0.6, -1.3, 1.1, -0.2, 1.6),
      c(1.7, -1.2, 0.9, -0.6, 0.4, -1.4)
    )
  )),
  config = list(
    graphType = "Heatmap", title = "Expression Heatmap",
    samplesClustered = TRUE, variablesClustered = TRUE
  ),
  height = 360
)

## 4. Straight from a `data.frame`

Usually your data is already a `data.frame` (or matrix), not a hand-written `y` object. `show_df()`
converts one for you: **row names become `vars`, column names become `smps`**, and the cell values
become the data matrix. That layout is exactly what a Heatmap (and most CanvasXpress graph types)
expect — features in rows, samples in columns.

In [ ]:
show_df <- function(df, config, ...) {
  df <- as.data.frame(df)
  data <- list(y = list(
    vars = as.list(rownames(df)),
    smps = as.list(colnames(df)),
    data = lapply(seq_len(nrow(df)), function(i) unlist(df[i, ], use.names = FALSE))
  ))
  show(data, config, ...)
}

# A plain data.frame: genes in rows, samples in columns.
expr <- data.frame(
  S1 = c(1.2, -0.9, 0.5, 1.7),
  S2 = c(-0.4, 1.4, 0.6, -1.2),
  S3 = c(0.8, 0.2, -1.3, 0.9),
  S4 = c(1.9, -0.7, 1.1, -0.6),
  S5 = c(-1.1, 1.8, -0.2, 0.4),
  S6 = c(0.3, -0.5, 1.6, -1.4),
  row.names = c("Gene A", "Gene B", "Gene C", "Gene D")
)

show_df(expr, list(
  graphType = "Heatmap", title = "Expression Heatmap (from a data.frame)",
  samplesClustered = TRUE, variablesClustered = TRUE
), height = 360)